# The placement walk

The headline claim of this project is that where a perturbation is injected dominates what the perturbation does. This notebook reads that claim off disk, the same way `scripts/paper/tab_staircase.py` and `scripts/paper/fig_basis_ranking.py` do, one axis at a time: first the residual stream against the paper's own site, then dropout moved across every site the network offers, then the operator held fixed while the site changes, then the site held fixed while the perturbation changes, then a band of blocks against all of them. Every number below comes from `results/<folder>/psbd_metrics.json`, written by `cli.analyze`, and nothing here recomputes a sweep.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in the library resolves the
# same way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import logging

logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import scripts.paper._style  # noqa: F401  the figure style every table in the paper uses

from cli.compare import detectors_psbd_values as psbd_values
from defences.decision import PUBLISHED_PLACEMENT, RECOMMENDED_PLACEMENT
from scripts.paper._common import (
    HEADLINE_KEY,
    bootstrap_ci,
    clearing_cells,
    load_coverage,
    load_declaration,
    load_psbd_metrics,
    mean_or_none,
)

RESULTS_DIR = "results"
coverage = load_coverage(RESULTS_DIR)
declaration = load_declaration("configs/psbd_basis.json")
print(f"basis declares {len(declaration['basis'])} placements")
print(f"coverage ledger holds {len(coverage['cells'])} models, "
      f"{len(clearing_cells(coverage))} clear the attack-success bar")

basis declares 24 placements
coverage ledger holds 105 models, 71 clear the attack-success bar


## The model set this notebook reads

A model enters the tables below only if both named placements, PSBD-TM and PSBD-RD, were swept and analyzed on it at both the adaptive and the matched rule, exactly the filter `tab_staircase.py` applies. The paper's own headline table reads this same filter at a fixed commit, 65 models. Results keep accumulating between that commit and whenever this notebook runs, so the count read here can be larger without the ranking changing.

In [2]:
def panel_cells(results_dir, coverage):
    """Clearing cells where PSBD-TM and PSBD-RD both reached the adaptive rule."""
    selected = []
    for cell in clearing_cells(coverage):
        report = load_psbd_metrics(results_dir, cell["folder_name"])
        if (
            psbd_values(report, RECOMMENDED_PLACEMENT, "adaptive") is not None
            and psbd_values(report, RECOMMENDED_PLACEMENT, "matched") is not None
            and psbd_values(report, PUBLISHED_PLACEMENT, "adaptive") is not None
        ):
            cell = dict(cell, report=report)
            selected.append(cell)
    return selected


cells = panel_cells(RESULTS_DIR, coverage)
print(f"{len(cells)} models carry both named placements at the adaptive rule")

69 models carry both named placements at the adaptive rule


## Reading one placement's adaptive-rule numbers over the model set

AUROC and the true-positive rate at two false-positive budgets, averaged over the models a placement was actually swept on. A placement missing from part of the model set prints its numbers at whatever count it covers rather than being dropped.

In [3]:
def measure_placement(cells, placement_id):
    auroc, tpr_10, tpr_20 = [], [], []
    for cell in cells:
        block = psbd_values(cell["report"], placement_id, "adaptive")
        if block is None:
            continue
        auroc.append(block[HEADLINE_KEY]["auroc"])
        if "q0.10" in block:
            tpr_10.append(block["q0.10"]["tpr"])
        if "q0.20" in block:
            tpr_20.append(block["q0.20"]["tpr"])
    return {
        "n": len(auroc),
        "auroc": mean_or_none(auroc),
        "tpr_at_10_percent": mean_or_none(tpr_10),
        "tpr_at_20_percent": mean_or_none(tpr_20),
        "below_chance": sum(1 for value in auroc if value < 0.5),
    }


def paired_gain(cells, placement_id, reference_id, resamples=5000, seed=0):
    """Mean AUROC gain, paired within model, with a bootstrap interval."""
    deltas = []
    for cell in cells:
        value = psbd_values(cell["report"], placement_id, "adaptive")
        reference = psbd_values(cell["report"], reference_id, "adaptive")
        if value is not None and reference is not None:
            deltas.append(value[HEADLINE_KEY]["auroc"] - reference[HEADLINE_KEY]["auroc"])
    low, high = bootstrap_ci(deltas, resamples, seed)
    return mean_or_none(deltas), low, high


def staircase_table(cells, rows, reference_id):
    """1 table: every row is (label, placement id), the first row is its own reference."""
    table_rows = []
    for label, placement_id in rows:
        measured = measure_placement(cells, placement_id)
        if placement_id == reference_id:
            gain_text = "--"
        else:
            gain, low, high = paired_gain(cells, placement_id, reference_id)
            gain_text = f"{gain:+.3f} [{low:+.3f}, {high:+.3f}]" if gain is not None else "--"
        table_rows.append({"placement": label, "gain": gain_text, **measured})
    return pd.DataFrame(table_rows).set_index("placement")

## 1. The residual stream against the paper's own site

The ResNet placement has no literal counterpart in a ViT block, which has two residual adds and no activation after them. This reads it as dropout after both adds, `post_residual`, and compares it against dropout confined to the attention add alone and dropout moved before both adds, whole and banded.

In [4]:
RESIDUAL_ROWS = (
    ("dropout, after both residual adds (PSBD-RD)", "post_residual"),
    ("dropout, after the attention add only", "after_attention_residual"),
    ("dropout, before both residual adds", "pre_residual"),
    ("dropout, before both adds, blocks 1 to 4", "pre_residual_blocks_1_4"),
    ("dropout, before both adds, blocks 5 to 8", "pre_residual_blocks_5_8"),
    ("dropout, before both adds, blocks 9 to 12", "pre_residual_blocks_9_12"),
)
staircase_table(cells, RESIDUAL_ROWS, "post_residual").round(3)

,gain,n,auroc,tpr_at_10_percent,tpr_at_20_percent,below_chance
placement,,,,,,
"dropout, after both residual adds (PSBD-RD)",--,69,0.823,0.634,0.703,12
"dropout, after the attention add only","-0.017 [-0.039, +0.011]",9,0.816,0.534,0.665,1
"dropout, before both residual adds","+0.012 [-0.016, +0.040]",69,0.836,0.631,0.719,6
"dropout, before both adds, blocks 1 to 4","-0.022 [-0.064, +0.020]",69,0.802,0.548,0.681,6
"dropout, before both adds, blocks 5 to 8","+0.052 [+0.025, +0.082]",69,0.876,0.687,0.750,5
"dropout, before both adds, blocks 9 to 12","+0.027 [-0.012, +0.069]",46,0.880,0.721,0.784,2


## 2. Dropout moved across every site

The perturbation stays dropout. Only where it attaches changes.

In [5]:
DROPOUT_SITE_ROWS = (
    ("dropout, after both residual adds (PSBD-RD)", "post_residual"),
    ("dropout, before both residual adds", "pre_residual"),
    ("dropout, attention input", "before_attention_norm"),
    ("dropout, attention input after norm", "before_attention"),
    ("dropout, MLP input", "before_mlp_norm"),
    ("dropout, embedding output", "after_embedding"),
)
staircase_table(cells, DROPOUT_SITE_ROWS, "post_residual").round(3)

,gain,n,auroc,tpr_at_10_percent,tpr_at_20_percent,below_chance
placement,,,,,,
"dropout, after both residual adds (PSBD-RD)",--,69,0.823,0.634,0.703,12
"dropout, before both residual adds","+0.012 [-0.016, +0.040]",69,0.836,0.631,0.719,6
"dropout, attention input","+0.039 [-0.000, +0.086]",37,0.925,0.792,0.897,0
"dropout, attention input after norm","+0.064 [-0.018, +0.157]",9,0.897,0.707,0.825,0
"dropout, MLP input","-0.022 [-0.053, +0.019]",9,0.812,0.507,0.641,1
"dropout, embedding output","+0.039 [-0.027, +0.119]",9,0.873,0.645,0.732,0


## 3. The operator held fixed, then the site held fixed

First the site stays at the attention input while the operator changes. Then the operator stays at token masking while the site changes over the rest of the network. Token masking at the attention input, PSBD-TM, closes this table.

In [6]:
OPERATOR_ROWS = (
    ("token mask, attention input (PSBD-TM)", "before_attention_norm_token_mask"),
    ("channel mask, attention input", "before_attention_norm_channel_mask"),
    ("gaussian noise, attention input", "before_attention_norm_gaussian"),
    ("dropout, attention input", "before_attention_norm"),
    ("token mask, attention output before the add", "before_attention_residual_token_mask"),
    ("token mask, both sublayer inputs", "both_sublayer_inputs_token_mask"),
    ("token mask, MLP input", "before_mlp_norm_token_mask"),
    ("gaussian noise, MLP input after norm", "before_mlp_gaussian"),
    ("token mask, after the attention add", "after_attention_residual_token_mask"),
    ("channel mask, MLP neurons", "mlp_neurons_channel_mask"),
    ("gain scale, MLP norm output", "mlp_norm_out_gain_scale"),
    ("scale up, input pixels", "input_pixels_scale_up"),
    ("dropout, after both residual adds (PSBD-RD)", "post_residual"),
)
staircase_table(cells, OPERATOR_ROWS, "before_attention_norm_token_mask").round(3)

,gain,n,auroc,tpr_at_10_percent,tpr_at_20_percent,below_chance
placement,,,,,,
"token mask, attention input (PSBD-TM)",--,69,0.927,0.766,0.835,2
"channel mask, attention input","-0.046 [-0.084, -0.011]",69,0.881,0.682,0.765,2
"gaussian noise, attention input","-0.128 [-0.188, -0.073]",69,0.799,0.540,0.655,9
"dropout, attention input","-0.043 [-0.078, -0.001]",37,0.925,0.792,0.897,0
"token mask, attention output before the add","-0.002 [-0.028, +0.027]",66,0.933,0.809,0.887,0
"token mask, both sublayer inputs","-0.029 [-0.058, +0.000]",69,0.899,0.699,0.787,1
"token mask, MLP input","-0.106 [-0.151, -0.064]",69,0.821,0.502,0.682,3
"gaussian noise, MLP input after norm","-0.056 [-0.100, -0.014]",69,0.872,0.703,0.783,8
"token mask, after the attention add","-0.162 [-0.217, -0.109]",69,0.765,0.443,0.577,13


Moving the operator at a fixed site costs far less than moving the site at a fixed operator, and dropout at the attention input already beats PSBD-RD by a wide margin, which says the site carries more of the effect than the perturbation does. Notebook 04 works through why token masking still adds a further gain at that same site.

## 4. Banding the winner to a range of blocks

Everything above perturbs every block. This restricts token masking at the attention input to three bands of four blocks each.

In [7]:
BAND_ROWS = (
    ("token mask, attention input, all 12 blocks (PSBD-TM)", "before_attention_norm_token_mask"),
    ("token mask, attention input, blocks 1 to 4", "before_attention_norm_blocks_1_4_token_mask"),
    ("token mask, attention input, blocks 5 to 8", "before_attention_norm_blocks_5_8_token_mask"),
    ("token mask, attention input, blocks 9 to 12", "before_attention_norm_blocks_9_12_token_mask"),
)
staircase_table(cells, BAND_ROWS, "before_attention_norm_token_mask").round(3)

,gain,n,auroc,tpr_at_10_percent,tpr_at_20_percent,below_chance
placement,,,,,,
"token mask, attention input, all 12 blocks (PSBD-TM)",--,69,0.927,0.766,0.835,2
"token mask, attention input, blocks 1 to 4",--,0,NaN,NaN,NaN,0
"token mask, attention input, blocks 5 to 8","-0.063 [-0.115, -0.021]",44,0.883,0.758,0.838,3
"token mask, attention input, blocks 9 to 12","-0.050 [-0.093, -0.012]",31,0.893,0.761,0.827,0


Restricting the winning placement to any single band costs mean AUROC against acting in every block. The backdoor is not written at one depth, so a probe confined to one band misses whatever the network wrote outside it, a point notebook 04 traces directly through the network's depth.

## The full basis, ranked

Every placement the basis declares, ranked by mean AUROC at the adaptive rule, restricted to placements measured on the same full count of models so the ranking compares like against like. This is `scripts/paper/fig_basis_ranking.py`'s figure, drawn here from the same reader it uses.

In [8]:
from scripts.paper.app_basis import ranking_rows

_rows, ordered = ranking_rows(RESULTS_DIR, clearing_cells(coverage), declaration["basis"])
full_count = max(ordered[p]["adaptive_n"] for p in ordered)
complete = [p for p in ordered if ordered[p]["adaptive_n"] == full_count]
ranked = complete[::-1]

figure, axis = plt.subplots(figsize=(7, 5.2))
values = [ordered[p]["adaptive_mean"] or 0.0 for p in ranked]
axis.barh(range(len(ranked)), values, height=0.6)
axis.set_yticks(range(len(ranked)))
axis.set_yticklabels(ranked, fontsize=7)
axis.axvline(0.5, color="black", lw=0.8)
axis.set_xlim(0.4, 1.0)
axis.set_xlabel("mean AUROC, adaptive rule")
figure.tight_layout()
plt.show()

print(f"placements measured on the full {full_count}-model panel: {len(ranked)}")
print(f"top: {ranked[-1]}, bottom: {ranked[0]}")

placements measured on the full 71-model panel: 12
top: both_sublayer_inputs_token_mask, bottom: input_pixels_scale_up


## What the walk establishes

Position and operator both move the result, and neither reduces to the other. The site does most of the work, since a fixed operator moved across sites spans a far wider range than a fixed site moved across operators, but the operator is not free either, since token masking beats dropout and gaussian noise at the same site by a wide margin. Acting in every block beats any single band. Notebook 04 explains this ranking by tracing where a trigger sits inside the network and how attention moves it toward the class token.